# 01 - Extract Hard Negatives (3-slice)

Variante de l'étape 01 du pipeline `colab_msd_recall` où chaque patch sauvé est un stack 3 canaux `(slice-1, slice, slice+1)` au lieu d'une grayscale répliquée. Le dataset résultant est dans `data/classifier_dataset_3slice/` et n'écrase pas le dataset 2D existant.

Exécuter ce notebook avec une GPU runtime.

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'clean-structure',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
else:
    print('Local run — data/ et work_dirs/ doivent déjà être peuplés.')

→ detect GPU
✓ deps already installed, skipping
✓  /content/INF8225_Projet/data already linked
✓  /content/INF8225_Projet/work_dirs already linked
✓  /content/INF8225_Projet/MedSAM/work_dir already linked
✓  /content/INF8225_Projet/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth already linked

Project : /content/INF8225_Projet
Drive   : /content/drive/MyDrive/Projet_Medsam
Device  : Tesla T4
Torch   : 2.4.0+cu121

⚠  numpy was already loaded in this kernel before setup reinstalled it.
   Runtime → Restart session, then run your imports again
   (no need to rerun setup — deps are pinned on disk).
cwd: /content/INF8225_Projet


In [ ]:
#@title Verify shared assets
from pathlib import Path

required = [
    Path("data/MSD_pancreas/train.json"),
    Path("data/MSD_pancreas/val.json"),
    Path("data/MSD_pancreas/test.json"),
    Path("data/MSD_pancreas/train/images"),
    Path("data/MSD_pancreas/val/images"),
    Path("data/MSD_pancreas/test/images"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("msd_implementation/configs/grounding_dino/pancreas_tumor.py"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"

OK      data/MSD_pancreas/train.json
OK      data/MSD_pancreas/val.json
OK      data/MSD_pancreas/test.json
OK      data/MSD_pancreas/train/images
OK      data/MSD_pancreas/val/images
OK      data/MSD_pancreas/test/images
OK      work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth
OK      work_dirs/tumor_config_v3/tumor_config_v3.py


In [ ]:
#@title Run pipeline step (3-slice extraction)
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.three_slice_context.extract_hard_negatives"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'experiments.three_slice.extract_hard_negatives_3slice'], returncode=0)

In [ ]:
#@title Inspect generated 3-slice classifier dataset
from pathlib import Path
base = Path("outputs/msd_implementation/three_slice_context/datasets/classifier_dataset_three_slice")
for split in ["train", "val"]:
    for cls in ["0", "1"]:
        folder = base / split / cls
        count = len(list(folder.glob("*.png"))) if folder.exists() else 0
        print(f"{split}/{cls}: {count} patches")
assert (base / "train" / "0").exists(), "classifier_dataset_3slice was not created"

train/0: 7203 patches
train/1: 2523 patches
val/0: 911 patches
val/1: 306 patches
